# Notebook 23 — Building Agents with Hugging Face smolagents

    ## Learning objectives

    - Rebuild the explicit agent loop with ToolCallingAgent and understand every abstraction
- Compare structured tool calling with code agents, planning, memory, and managed agents
- Connect local Transformers, Ollama, or vLLM inference without hiding security boundaries

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['smolagents>=1.17', 'huggingface-hub>=0.30,<1', 'python-dotenv>=1.1']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 23.1 Why use a framework after building the loop yourself?

Notebook 22 made the control loop explicit: the model proposes an action, trusted code validates and executes it,
the observation returns to context, and deterministic limits decide whether another step is allowed. An agent
framework packages this state machine, tool schemas, prompting, memory, streaming, and model adapters. It saves
implementation work but does not own your authorization, data policy, or product correctness.

Hugging Face `smolagents` fits this course because it is open, compact, and supports both structured tool calling
and code-producing agents. Treat its APIs as one concrete implementation of general concepts. Pin the version;
inspect generated prompts and traces; and keep business logic outside framework-specific message objects.


In [ ]:
import importlib.util, json, os
SMOLAGENTS_AVAILABLE = importlib.util.find_spec("smolagents") is not None
print("smolagents installed:", SMOLAGENTS_AVAILABLE)
# Colab setup installs it. Locally: uv add smolagents, or keep reading the executable contracts below.


## 23.2 Typed tools are contracts

A tool name and description influence selection; type hints and argument descriptions constrain generation; the
implementation defines the real capability. Make tools narrow, deterministic where possible, and explicit about
units, allowed identifiers, errors, and side effects. Validate again inside the implementation. Do not expose a
generic shell, arbitrary URL fetch, or unrestricted SQL interface because a schema makes it look tidy.

`smolagents.tool` converts a typed, documented function into a tool. Production wrappers should add authenticated
caller context outside model-controlled arguments, idempotency keys for retryable mutations, deadlines, output
limits, structured audit events, and redaction. The model must never choose its own tenant or permission scope.


In [ ]:
if SMOLAGENTS_AVAILABLE:
    from smolagents import tool
    @tool
    def multiply(a: int, b: int) -> int:
        """Multiply two bounded integers.

        Args:
            a: First integer between -10000 and 10000.
            b: Second integer between -10000 and 10000.
        """
        if not all(-10_000 <= value <= 10_000 for value in (a, b)): raise ValueError("out of range")
        return a * b
    print(multiply.name, multiply.description, multiply.inputs)
else:
    print("Expected schema: multiply(a: integer, b: integer) -> integer")


## 23.3 ToolCallingAgent versus CodeAgent

`ToolCallingAgent` asks a capable chat model for structured calls. It is a good default when actions fit a small
schema and every call should be inspectable. `CodeAgent` expresses actions as code, which can compose loops,
transformations, and multiple tool calls efficiently. Code is also a much larger capability: local execution can
read files, consume resources, or escape assumptions unless isolated.

Prefer structured calls for consequential business operations. Use code agents for analysis when a sandbox has
explicit filesystem/network/import/resource limits and disposable state. `smolagents` exposes executor choices,
but naming an executor “sandboxed” is not enough; threat-model its host, credentials, mounts, network, kernel,
timeouts, and output channels.


In [ ]:
agent_choice = {
    "send_invoice": "ToolCallingAgent: typed, approved, idempotent mutation",
    "analyze_local_table": "CodeAgent in an isolated executor",
    "lookup_order": "ToolCallingAgent: narrow read-only operation",
    "arbitrary_shell": "Do not expose; replace with task-specific tools",
}
for task, choice in agent_choice.items(): print(f"{task:24} -> {choice}")


## 23.4 Model adapters and a guarded agent

Agent quality depends on the model's exact chat template and tool-use training. A general text model may emit
malformed calls even when it writes fluent answers. `smolagents` can connect to Hugging Face inference and local
OpenAI-compatible servers; keeping a model adapter narrow makes it possible to switch among local Transformers,
Ollama, and vLLM after conformance tests.

The following construction is opt-in because it may perform remote inference. Credentials come from Colab Secrets
or `.env`, never notebook literals. Begin with read-only tools, `max_steps`, deterministic decoding, and trace
inspection. Production also needs total token/time/cost budgets and cancellation propagation.


In [ ]:
RUN_AGENT = False
if RUN_AGENT and SMOLAGENTS_AVAILABLE:
    from smolagents import InferenceClientModel, ToolCallingAgent
    model = InferenceClientModel(model_id=os.getenv("HF_AGENT_MODEL", "Qwen/Qwen2.5-7B-Instruct"),
                                 token=os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN"))
    agent = ToolCallingAgent(tools=[multiply], model=model, max_steps=4, planning_interval=2)
    answer = agent.run("Use the tool to compute 137 times 42, then explain the result briefly.")
    print(answer)
    agent.memory.replay(detailed=False)
else:
    print("Agent run skipped. Configure credentials/model and opt in after reviewing the tool boundary.")


## 23.5 Planning, memory, and managed agents

Planning intervals ask the model to periodically reconsider progress. Planning can help long tasks but consumes
tokens and may produce confident fiction. A plan is working state, not authorization. Agent memory records task,
action, observation, error, and planning steps for the current run; it should be bounded and redacted before
persistence. Summaries can omit constraints, so retain authoritative structured state separately.

Managed agents let a coordinator delegate to specialists described like tools. Define typed task/result contracts,
isolate tools and credentials, cap depth and fan-out, and record lineage. Do not share entire histories by default.
Notebook 26 develops multi-agent workflow patterns and demonstrates why delegation is an architectural cost, not
a free accuracy multiplier.


In [ ]:
trace = [
    {"step":0, "kind":"task", "content":"compute and explain"},
    {"step":1, "kind":"action", "tool":"multiply", "arguments":{"a":137,"b":42}},
    {"step":1, "kind":"observation", "content":5754},
    {"step":2, "kind":"final", "content":"137 × 42 = 5754."},
]
allowed_trace_fields = {"step", "kind", "tool", "arguments", "content"}
print(json.dumps([{k:v for k,v in event.items() if k in allowed_trace_fields} for event in trace], indent=2))


## 23.6 Testing and portability

Unit-test tools without a model. Contract-test schemas against the model template. Replay frozen model actions into
fake tools. Inject timeouts, malformed results, unavailable dependencies, duplicate requests, and permission
denials. Evaluate final success plus tool selection, arguments, unnecessary calls, step count, recovery, latency,
and side effects. Never let nondeterministic live APIs make the only CI test.

Exporting or serializing an agent can capture prompts and tool code, but deployments still need dependency locks,
model/template revisions, secret injection, network policy, permissions, and monitoring. Treat Hub artifacts as
untrusted code/data until reviewed. Framework upgrades are release changes because default prompts, parsing, memory,
and model adapters can alter behavior.


## Exercises

    1. Implement the same bounded task with the custom loop and ToolCallingAgent; compare traces.
2. Create a CodeAgent threat model and a sandbox policy before enabling execution.
3. Build a fake model adapter that returns malformed, repeated, and unauthorized calls for tests.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
